# 초구 예측 — model_15 (3단계: 위치에 타자 장타 비율)

- model_14: 위치 상황 보정 채택 (README 4-29)
- 0장 0-1: model_14 위 타자 성향 -> 묶음 전체는 0, 존 밖 스윙률은 나빠짐, **장타 비율만** 두 방식 모두 + (+1.06 / +0.81 ×1000)
- 모델: model_14 위치 보정 LightGBM에 **타자 장타 비율(`bat_xbh`)만** 더함 (사용자 결정 (가): 만들어서 2020까지 확인)
  - 장타 비율 = 그 타자 전날까지 타석당 2루타·3루타·홈런 비율, 리그 쪽으로 50타석 당김 (`fc.batter_location_features`)
  - LightGBM 파라미터·트리 수를 2019로 다시 고름 (변수가 바뀌어 연결됨), 나머지는 model_14 그대로
- 비교 대상: model_14

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_15"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

### 0-1. model_14 위 타자 성향

- 구조: model_14과 같음 (model_13 위치 log 확률을 출발점으로 24칸 LightGBM, 학습 2018 -> 2019, 파라미터 잎 7·잎당 최소 20·λ 1000)
- 타자 성향 변수 (전날까지, `fc.opponent_features`·`fc.batter_location_features`, 미래 정보 검사 통과 4-28)
  - `bat_swing` 초구 스윙률, `bat_chase` 존 밖 공 스윙률, `bat_zswing` 존 안 공 스윙률, `bat_xbh` 타석당 장타 비율 (모두 리그 쪽으로 당긴 비율)
- 비교 두 가지 (이득 ×1000 = model_14 구조 − 후보, 2019 24칸 log loss, 경기 단위 95% 구간)
  - **트리 수 고정 168** (model_14과 같은 설정, 기준 = model_14 그대로 재현)
  - **트리 수 조기 종료** (2018 경기 20%로, 0-3 방식) -> 변수가 늘면 트리 수가 달라질 수 있어서
- 묶음 전체 + 하나씩 (어떤 타자 변수가 남는지)

In [2]:
DIAG = "model_14"
pp18 = fc.load_results([DIAG])[DIAG]
LOC_REF, LOC_REF_HL = "decay", 173.0  # model_13 위치 정규분포 (model_14과 같음)
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 30.0
LOC_SEASON_F = 0.707
LOC_AC = (None, 12.5, 17.3)
LOC_VAR = (200.0, 9.74, False, 316.0)
ZONE_M = 32
LOC_SIT = ["base1", "base2", "base3", "first_open", "sdiff", "inning", "out_cn", "home", "batorder"]  # model_14 상황 변수
LOC_FEAT18 = CLASSES + ["same_side"] + LOC_SIT
LOC_LGB18, LOC_ROUNDS18 = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}, 168  # model_14 LightGBM
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트 (좌우, 높이 비율, 제곱, 곱)
_MO_MOM, _WT = {}, {}

def mo_mom(ref_hl):  # 다른 카운트 잔차 모멘트 표 (x = 좌우 잔차, zn = 높이 비율 잔차, 제곱·곱)
    if ref_hl not in _MO_MOM:
        d = mo_resid(ref_hl)
        _MO_MOM[ref_hl] = d.assign(xx=d["rx"] ** 2, zz=d["rz"] ** 2, xz=d["rx"] * d["rz"]).rename(columns={"rx": "x", "rz": "zn"})
    return _MO_MOM[ref_hl]

def within_tab(name, data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) 표와 전체 값
    if (name, year) not in _WT:
        past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
        g = past.groupby(K2)
        n = g.size()
        w = (n - 1).clip(lower=0)
        c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                          "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * n / (n - 1).clip(lower=1)}).fillna(0)  # 표본 분산·공분산
        num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
        wg = w.groupby(level=["pitch_type", "same_side"]).sum()  # 구종×손별 자유도 합 (표본 크기)
        _WT[(name, year)] = (num.div(wg, axis=0), c.mul(w, axis=0).sum() / w.sum(), wg)
    return _WT[(name, year)]

def v0_of(q, name, data, year, k0=0.0):  # 공마다 그 구종×손 투수 안 분산·공분산 (n, 3): 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김
    tab, allv, wg = within_tab(name, data, year)
    idx = pd.MultiIndex.from_frame(q[KR])
    a = allv.to_numpy()[None, :]  # 전체 구종 투수 안 분산
    v = tab.reindex(idx).to_numpy()
    n = wg.reindex(idx).fillna(0).to_numpy()[:, None]  # 그 구종×손 자유도 합
    v = np.where(np.isnan(v), a, v)
    return np.where(n + k0 > 0, (n * v + k0 * a) / np.maximum(n + k0, 1e-12), a)  # 예: 너클 초구 3구(자유도 2)면 거의 전체 값

def own_var(q, year, ref_hl, hl_var, season_f, k0=0.0, cache=True):  # 공마다 다른 카운트 퍼짐 -> 초구 단위로 바꾼 자기 분산·공분산 (n, 3), (가중) 공 수 (n,), V0 (n, 3)
    data = mo_mom(ref_hl)
    S, N = fc.lookup(fc.prior_table(data, K2, MCOLS, hl_var, season_f=season_f, cache=cache), q, K2, MCOLS, season_f=season_f)
    e = S / np.maximum(N, 1e-12)
    v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 잔차 분산·공분산
    V0 = v0_of(q, "초구", fp, year, k0)
    V0m = v0_of(q, ("다른 카운트", ref_hl), data, year, k0)
    r = np.sqrt(np.maximum(v[:, :2], 0) / V0m[:, :2])  # 좌우·높이 SD 비율 (자기 / 투수 안)
    v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0m[:, 0] * V0m[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
    n = N[:, 0]
    return np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n, V0

def apply_var(M, v, n, V0, k_var, unc, N2, k_hand):  # 모멘트 M의 분산·공분산을 따로 당긴 값으로 바꿈 (평균 그대로)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 k_var만큼
    if unc:
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    M[:, 2], M[:, 3], M[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return M

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None):  # 위치 모멘트 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # 모든 카운트 위치 성향 (model_12)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])
        P = shift_mean(P, d[:, 0], d[:, 1])
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)  # 투수×구종 -> 투수×구종×손
    if var is not None:  # 분산 따로 (model_13): var = (hl_var, k_var, unc, k0)
        v, n, V0 = own_var(q, year, None if ac is None else ac[0], var[0], season_f, var[3])
        M = apply_var(M, v, n, V0, var[1], var[2], N2, k_hand)
    return M

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac, var), qt)
    return zone_probs(g, qt, cov), g



LOC17 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC, var=LOC_VAR)
PZ_TRUE = np.full((len(fp), len(ZONES)), np.nan)  # 실제 구종 기준 model_13 24칸 확률
for yr in (2018, fc.VAL_YEAR):
    k = fp["GYEAR"].eq(yr).to_numpy()
    PZ_TRUE[k] = loc_given_type(fp[k], yr, fp.loc[k, "pitch_type"].to_numpy(), **LOC17)[0]

from scipy.special import log_softmax
SIT_L = fc.situation_features(m, fp)
XLOC = pd.concat([fp[CLASSES + ["same_side"]], SIT_L, fc.opponent_features(m, fp)[["home", "bat_swing"]], fc.batter_location_features(m, fp)], axis=1)
XLOC["first_open"] = (((SIT_L["base2"] + SIT_L["base3"]) > 0) & SIT_L["base1"].eq(0)).astype(int)
BAT = ["bat_swing", "bat_chase", "bat_zswing", "bat_xbh"]  # 타자 성향
NZ = len(ZONES)
INIT_L = np.log(np.clip(PZ_TRUE, 1e-6, None))
INIT_L -= INIT_L.mean(1, keepdims=True)
yr_l = fp["GYEAR"].to_numpy()
BASE_L = dict(objective="multiclass", num_class=NZ, learning_rate=0.05, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
              verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_14과 같음
itr = np.flatnonzero(yr_l == 2018)
i19 = np.flatnonzero(val)
gid_all = fp["gameID"].to_numpy()
g18 = np.unique(gid_all[itr])
es = np.isin(gid_all[itr], np.random.default_rng(1).choice(g18, len(g18) // 5, replace=False))  # 조기 종료용 경기 20%

def fit_pred(cols, rounds=None):  # cols로 2018 학습 -> 2019 24칸 확률, 트리 수 (rounds 없으면 2018 경기 20%로 조기 종료)
    if rounds is None:
        t_, v_ = itr[~es], itr[es]
        dtr = lgb.Dataset(XLOC[cols].iloc[t_], y_zone[t_], init_score=INIT_L[t_], free_raw_data=False)
        dva = lgb.Dataset(XLOC[cols].iloc[v_], y_zone[v_], init_score=INIT_L[v_], reference=dtr)
        bst = lgb.train({**BASE_L, **LOC_LGB18}, dtr, 3000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
        n = bst.best_iteration
    else:
        bst = lgb.train({**BASE_L, **LOC_LGB18}, lgb.Dataset(XLOC[cols].iloc[itr], y_zone[itr], init_score=INIT_L[itr]), rounds)
        n = rounds
    raw = bst.predict(XLOC[cols].iloc[i19], raw_score=True, num_iteration=n) + INIT_L[i19]
    return np.exp(log_softmax(raw, axis=1)), n

y19 = y_zone[i19]
oc19 = (fc.ZONE_GROUP[y19] == "C").astype(float)
nll19 = lambda P: -np.log(np.clip(P[np.arange(len(i19)), y19], 1e-12, None))

def game_ci(d, B=1000):  # 경기 단위 부트스트랩 95% 구간 (2019)
    s = pd.DataFrame({"g": gid_all[i19], "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

def auc(p, o):  # Mann-Whitney AUC
    r = pd.Series(p).rank().to_numpy()
    n1, n0 = o.sum(), len(o) - o.sum()
    return (r[o == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

P18, _ = fit_pred(LOC_FEAT18, LOC_ROUNDS18)
assert np.allclose(nll19(P18), fc.align(pp18, fp, fc.VAL_YEAR)["ll_zone_true"].to_numpy(), atol=1e-6), "model_14 재현 실패"
print(f"model_14 2019 24칸 log loss (실제 구종) {nll19(P18).mean():.5f} (재현 확인)")
rows = []
for mode, rounds in [("트리 168 고정", LOC_ROUNDS18), ("조기 종료", None)]:
    Pb, nb_ = (P18, LOC_ROUNDS18) if rounds else fit_pred(LOC_FEAT18)
    llb = nll19(Pb)
    rows.append({"방식": mode, "더한 변수": "(model_14 구조)", "트리 수": nb_, "2019 24칸 ll": llb.mean(), "이득 ×1000": 0.0,
                 "존 밖 AUC": auc(Pb[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
    for nm, cols in [("타자 성향 전부", BAT)] + [(c, [c]) for c in BAT]:
        P, n_ = fit_pred(LOC_FEAT18 + cols, rounds)
        d = llb - nll19(P)
        lo, hi = game_ci(d)
        rows.append({"방식": mode, "더한 변수": nm, "트리 수": n_, "2019 24칸 ll": nll19(P).mean(), "이득 ×1000": d.mean() * 1000,
                     "95% 하한": lo * 1000, "95% 상한": hi * 1000, "존 밖 AUC": auc(P[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
display(pd.DataFrame(rows).set_index(["방식", "더한 변수"]).round(4))

model_14 2019 24칸 log loss (실제 구종) 2.88354 (재현 확인)


트리 수  2019 24칸 ll  이득 ×1000  존 밖 AUC  95% 하한  95% 상한
방식        더한 변수                                                              
트리 168 고정 (model_14 구조)   168       2.8835    0.0000   0.5546     NaN     NaN
          타자 성향 전부        168       2.8835    0.0471   0.5525 -1.4321  1.4923
          bat_swing       168       2.8837   -0.1662   0.5537 -0.9235  0.6240
          bat_chase       168       2.8850   -1.4656   0.5535 -2.2818 -0.6630
          bat_zswing      168       2.8837   -0.1919   0.5540 -1.0751  0.6432
          bat_xbh         168       2.8825    1.0606   0.5546  0.0696  2.0155
조기 종료     (model_14 구조)   201       2.8832    0.0000   0.5557     NaN     NaN
          타자 성향 전부        188       2.8837   -0.4814   0.5533 -1.9971  1.0583
          bat_swing       123       2.8835   -0.2712   0.5530 -1.1736  0.6770
          bat_chase       224       2.8849   -1.6684   0.5539 -2.4633 -0.7880
          bat_zswing      159       2.8835   -0.2794   0.5542 -1.3163  0.7662
          bat_xbh         230       2.8824    0.8136   0.5568 -0.1759  1.8927

#### 🔍 0-1 결과: 타자 성향 묶음은 model_14 위에 남는 몫 없음, 장타 비율만 조금 +
| 더한 변수 (이득 ×1000, 2019) | 트리 168 고정 (95%) | 조기 종료 (95%) |
| --- | --- | --- |
| 타자 성향 전부 | +0.05 (−1.43 ~ +1.49) | −0.48 (−2.00 ~ +1.06) |
| 초구 스윙률 `bat_swing` | −0.17 (−0.92 ~ +0.62) | −0.27 (−1.17 ~ +0.68) |
| 존 밖 스윙률 `bat_chase` | **−1.47** (−2.28 ~ −0.66) | **−1.67** (−2.46 ~ −0.79) |
| 존 안 스윙률 `bat_zswing` | −0.19 (−1.08 ~ +0.64) | −0.28 (−1.32 ~ +0.77) |
| 장타 비율 `bat_xbh` | **+1.06** (+0.07 ~ +2.02) | +0.81 (−0.18 ~ +1.89) |

- model_14 재현 확인 (2019 2.88354)
- 묶음 전체는 0: 0-3 사전 확인(상황 없이)에서 보였던 몫(+1.1)은 상황(타순 등)과 겹친 것으로 보임 (강타자 = 중심 타순)
- 존 밖 스윙률은 오히려 나빠짐 (두 방식 모두 95% 구간이 0 아래) -> 학습 해(2018)에서만 맞는 패턴
- 장타 비율만 두 방식 모두 + (+1.06 / +0.81), 한 방식은 95% 구간 0 위 -> 4개 중 하나만 겨우 + 라 우연일 가능성도 있음


## 모델

In [3]:
# 0장이 모델 변수 이름을 덮어쓰지 않았는지 대비: 설정 상수를 다시 정함
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]


## 1. 파라미터

- 구종은 model_09 값, 위치 정규분포는 model_13 값 고정
- 위치 보정 LightGBM: 변수 `LOC_FEAT` = model_14 + 장타 비율, 파라미터는 3-2에서 다시 고름

In [4]:
# ① 구종 출발점 B4: model_06 값 (고정) - 반감기·alpha·hand_k, 시즌 초 배수 (기준점 = 지난 시즌 전부)
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)
# ① LightGBM 변수: A 이번 타석 상황 + 앞 타석 초구 결과 원-핫 (fc.FIRST_CATS)
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]
# ② 위치 정규분포: model_13 값 (고정)
LOC_REF, LOC_REF_HL = "decay", 173.0  # 기준점: 분석 대상 투수 구종×손, 전날까지 반감 173구
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 30.0  # 투수 기록 반감기, 투수×구종 당김 k, 투수×구종×손 당김 k
LOC_SEASON_F = 0.707  # 시즌 초 배수
LOC_AC = (None, 12.5, 17.3)  # 모든 카운트 위치 성향 (잔차 기준 반감 None = 누적, δ 반감, δ 당김 k)
LOC_VAR = (200.0, 9.74, False, 316.0)  # 분산 따로 (다른 카운트 퍼짐 반감, k_var, unc, 투수 안 분산 당김 k0)
# ② 위치 상황 보정 (model_14): 24칸 LightGBM, 출발점 = model_13 위치 log 확률
LOC_SIT = ["base1", "base2", "base3", "first_open", "sdiff", "inning", "out_cn", "home", "batorder"]  # 상황 변수 (0-3 묶음 4)
LOC_FEAT = CLASSES + ["same_side"] + LOC_SIT + ["bat_xbh"]  # 구종 원-핫·타자 손 + 상황 + 타자 장타 비율 (model_15)
LOC_TRAIN_FIRST = 2018  # 위치 보정 학습 첫 해 (2017은 model_13 위치 없음)
ZONE_M = 32
# ① LightGBM: model_07 값 (고정)
LGB_P = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}
LGB_ROUNDS = 730
RETRAIN = "month"  # 재학습 주기: 그 달 첫 경기 전날까지 데이터로 다시 학습 (3월은 4월에, 10월은 9월에 합침)
# 구종 보정 학습 해
TRAIN_FIRST = 2017

## 2. ① 구종: B4 + LightGBM 월 단위 재학습 (model_09과 같음)

### 2-1. B4 (model_06·07·09과 같음)

In [5]:
def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06 B4 (n, 9): 지난 시즌 리그 비율 -> 투수 (alpha) -> 투수×손 (hand_k), 시즌 초 배수
    first = year == fp["GYEAR"].min()  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)  # 2017–2020
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
b4_19 = fc.prob_metrics(P_B4[val], y_type[val])["logloss"]
assert abs(b4_19 - 1.09367) < 1e-5, b4_19  # model_06·07 B4의 2019 log loss
print(f"2019 B4 log loss {b4_19:.5f} = model_06·07 B4 (재현 확인)")
m07 = pd.read_parquet("results/model_07/per_pitch.parquet")  # 첫 달 재현 확인용
m09 = pd.read_parquet("results/model_09/per_pitch.parquet")  # 구종 재현 확인용 (구종은 model_09 그대로)
m13 = pd.read_parquet("results/model_13/per_pitch.parquet")  # 위치 정규분포 재현 확인용
m14 = pd.read_parquet("results/model_14/per_pitch.parquet")  # 비교 대상

2019 B4 log loss 1.09367 = model_06·07 B4 (재현 확인)


### 2-2. 변수 (model_07과 같음)

- A 이번 타석 상황 + 앞 타석 초구 결과, 미래 정보 검사 다시 통과 확인

In [6]:
X_all = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)[FEAT]
assert X_all.notna().all().all()
for fn in (fc.situation_features, fc.game_flow_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=150)
    assert bad.empty, bad
display(X_all.describe().T[["mean", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,mean,min,max
pc,33.394,0.0,124.0
tto,1.592,1.0,5.0
starter,0.690,0.0,1.0
inning,4.548,1.0,12.0
out_cn,0.970,0.0,2.0
base1,0.359,0.0,1.0
base2,0.194,0.0,1.0
base3,0.100,0.0,1.0
batorder,4.811,1.0,9.0
sdiff,0.369,-21.0,19.0


### 2-2. 월 단위 재학습과 구종 확률 (model_09과 같음)

- 확인: 구종 확률 = model_09 결과 파일 (재현)

In [7]:
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_05–07과 같음
yr_all = fp["GYEAR"].to_numpy()
DATE = pd.to_datetime(fp["gameID"].str[:8]).to_numpy()
MON = np.clip(fp["gameID"].str[4:6].astype(int).to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 LightGBM (트리 수 고정)
    return lgb.train({**BASE, **LGB_P}, lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr]), LGB_ROUNDS)

def type_prob(bst, idx):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    return np.exp(log_softmax(bst.predict(X_all.iloc[idx], raw_score=True) + INIT[idx], axis=1))

P_TYPE_ALL = np.full((len(fp), K), np.nan)
BST, plan = {}, []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_all == yr)
    for mon in np.unique(MON[iy]):
        te = iy[MON[iy] == mon]
        start = DATE[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_all >= TRAIN_FIRST) & (DATE < start))
        assert DATE[tr].max() < start  # 미래 정보 없음
        BST[(yr, mon)] = bst = fit_rows(tr)
        P_TYPE_ALL[te] = type_prob(bst, te)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_all[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))
P11 = {yr: fc.align(m07, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # model_07 구종 확률 (fp 그 해 순서)
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:  # 첫 달 = 시즌 단위 (model_07 재현)
    k = MON[msk] == MON[msk].min()
    assert np.allclose(P_TYPE_ALL[msk][k], P11[yr][k], atol=1e-6)
print("첫 달 구종 확률 = model_07 (재현 확인)")
rows = {}
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("B4", P_B4[msk]), ("model_07", P11[yr]), (MODEL, P_TYPE_ALL[msk])]:
        r = fc.prob_metrics(P, y_type[msk])
        rows[(yr, nm)] = {"log loss": r["logloss"], "top1": r["top1"], "top2": r["top2"]}
display(pd.DataFrame(rows).T.round(4))
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    assert np.allclose(P_TYPE_ALL[msk], fc.align(m09, fp, yr)[fc.P_TYPE].to_numpy(), atol=1e-6), f"{yr} 구종 확률이 model_09과 다름"
print("구종 확률 = model_09 (재현 확인)")

예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  24636           0  2018-10-14  2019-03-23
     5   2403  27204        2568  2019-04-30  2019-05-01
     6   2493  29607        4971  2019-05-31  2019-06-01
     7   1791  32100        7464  2019-06-30  2019-07-02
     8   2362  33891        9255  2019-07-31  2019-08-01
     9   1841  36253       11617  2019-08-31  2019-09-01
2020 5   2264  38094           0  2019-10-01  2020-05-05
     6   2379  40358        2264  2020-05-31  2020-06-02
     7   2233  42737        4643  2020-06-30  2020-07-01
     8   1985  44970        6876  2020-07-31  2020-08-01
     9   3754  46955        8861  2020-08-30  2020-09-01

첫 달 구종 확률 = model_07 (재현 확인)


log loss    top1    top2
2019 B4          1.0937  0.5279  0.8159
     model_07    1.0621  0.5380  0.8174
     model_15    1.0613  0.5385  0.8177
2020 B4          1.0916  0.5264  0.8179
     model_07    1.0570  0.5340  0.8209
     model_15    1.0561  0.5332  0.8218

구종 확률 = model_09 (재현 확인)


## 3. ② 위치: model_13 정규분포 + 상황·장타 비율 보정

### 3-1. model_13 위치 (그대로)

- 위치 함수는 model_13 3장 복사 -> 실제 구종 기준 24칸 log loss가 model_13 결과 파일과 같은지 확인

In [9]:
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트 (좌우, 높이 비율, 제곱, 곱)
_MO_MOM, _WT = {}, {}

def mo_mom(ref_hl):  # 다른 카운트 잔차 모멘트 표 (x = 좌우 잔차, zn = 높이 비율 잔차, 제곱·곱)
    if ref_hl not in _MO_MOM:
        d = mo_resid(ref_hl)
        _MO_MOM[ref_hl] = d.assign(xx=d["rx"] ** 2, zz=d["rz"] ** 2, xz=d["rx"] * d["rz"]).rename(columns={"rx": "x", "rz": "zn"})
    return _MO_MOM[ref_hl]

def within_tab(name, data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) 표와 전체 값
    if (name, year) not in _WT:
        past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
        g = past.groupby(K2)
        n = g.size()
        w = (n - 1).clip(lower=0)
        c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                          "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * n / (n - 1).clip(lower=1)}).fillna(0)  # 표본 분산·공분산
        num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
        wg = w.groupby(level=["pitch_type", "same_side"]).sum()  # 구종×손별 자유도 합 (표본 크기)
        _WT[(name, year)] = (num.div(wg, axis=0), c.mul(w, axis=0).sum() / w.sum(), wg)
    return _WT[(name, year)]

def v0_of(q, name, data, year, k0=0.0):  # 공마다 그 구종×손 투수 안 분산·공분산 (n, 3): 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김
    tab, allv, wg = within_tab(name, data, year)
    idx = pd.MultiIndex.from_frame(q[KR])
    a = allv.to_numpy()[None, :]  # 전체 구종 투수 안 분산
    v = tab.reindex(idx).to_numpy()
    n = wg.reindex(idx).fillna(0).to_numpy()[:, None]  # 그 구종×손 자유도 합
    v = np.where(np.isnan(v), a, v)
    return np.where(n + k0 > 0, (n * v + k0 * a) / np.maximum(n + k0, 1e-12), a)  # 예: 너클 초구 3구(자유도 2)면 거의 전체 값

def own_var(q, year, ref_hl, hl_var, season_f, k0=0.0, cache=True):  # 공마다 다른 카운트 퍼짐 -> 초구 단위로 바꾼 자기 분산·공분산 (n, 3), (가중) 공 수 (n,), V0 (n, 3)
    data = mo_mom(ref_hl)
    S, N = fc.lookup(fc.prior_table(data, K2, MCOLS, hl_var, season_f=season_f, cache=cache), q, K2, MCOLS, season_f=season_f)
    e = S / np.maximum(N, 1e-12)
    v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 잔차 분산·공분산
    V0 = v0_of(q, "초구", fp, year, k0)
    V0m = v0_of(q, ("다른 카운트", ref_hl), data, year, k0)
    r = np.sqrt(np.maximum(v[:, :2], 0) / V0m[:, :2])  # 좌우·높이 SD 비율 (자기 / 투수 안)
    v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0m[:, 0] * V0m[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
    n = N[:, 0]
    return np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n, V0

def apply_var(M, v, n, V0, k_var, unc, N2, k_hand):  # 모멘트 M의 분산·공분산을 따로 당긴 값으로 바꿈 (평균 그대로)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 k_var만큼
    if unc:
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    M[:, 2], M[:, 3], M[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return M

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None):  # 위치 모멘트 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # 모든 카운트 위치 성향 (model_12)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])
        P = shift_mean(P, d[:, 0], d[:, 1])
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)  # 투수×구종 -> 투수×구종×손
    if var is not None:  # 분산 따로 (model_13): var = (hl_var, k_var, unc, k0)
        v, n, V0 = own_var(q, year, None if ac is None else ac[0], var[0], season_f, var[3])
        M = apply_var(M, v, n, V0, var[1], var[2], N2, k_hand)
    return M

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac, var), qt)
    return zone_probs(g, qt, cov), g


LOC17 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC, var=LOC_VAR)
PZ_TRUE = np.full((len(fp), len(ZONES)), np.nan)  # 실제 구종 기준 model_13 24칸 확률 (2018–2020)
for yr in range(LOC_TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    PZ_TRUE[k] = loc_given_type(fp[k], yr, fp.loc[k, "pitch_type"].to_numpy(), **LOC17)[0]
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    ll_ = -np.log(np.clip(PZ_TRUE[msk][np.arange(msk.sum()), y_zone[msk]], 1e-12, None))
    assert np.allclose(ll_, fc.align(m13, fp, yr)["ll_zone_true"].to_numpy()), f"{yr} model_13 위치 재현 실패"
print("실제 구종 기준 24칸 = model_13 (재현 확인), 2018 계산된 공:", int(np.isfinite(PZ_TRUE[fp["GYEAR"].eq(2018).to_numpy()]).all(1).sum()))

실제 구종 기준 24칸 = model_13 (재현 확인), 2018 계산된 공: 12395


### 3-2. 위치 보정 LightGBM (2019로 다시 고르기)

- model_14과 같은 구조, 변수에 장타 비율만 더함 (`fc.batter_location_features`의 `bat_xbh`, 미래 정보 검사 다시 통과 확인)
- 학습 2018 -> 2019 조기 종료로 트리 수, 잎 수 · 잎당 최소 공 · lambda_l2 격자 (범위 끝이면 넓힘), 2019 1등과 0.0002 안이면 덜 극단적인 값
- 대조: model_14 (2019 2.88354)

In [10]:
def grid_search(score, grid, by="logloss"):  # grid의 모든 조합을 score로 채점 (지표 낮은 순)
    names_ = list(grid)
    combos = pd.MultiIndex.from_product(grid.values(), names=names_).to_frame(index=False)  # 후보의 모든 조합 (한 행 = 한 조합)
    rows = [{**r, **score(**r)} for r in combos.astype(object).where(combos.notna(), None).to_dict("records")]  # NaN -> None으로 되돌려 넘김
    return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

def finer(cands, best, n=3):
    # 거친 후보 목록에서 best와 양옆 후보 사이를 각각 로그 간격으로 n개씩 나눈 세밀한 후보 (best 가운데, 2n−1개)
    # best가 범위 끝이면 바깥으로 2배 넓힘, best가 None이면 None 그대로
    if best is None:
        return [None]
    nums = sorted(c for c in cands if c is not None)  # None 뺀 숫자 후보
    i = nums.index(best)
    lo = nums[i - 1] if i > 0 else best / 2  # 왼쪽 이웃 (없으면 절반)
    hi = nums[i + 1] if i < len(nums) - 1 else best * 2  # 오른쪽 이웃 (없으면 2배)
    vals = np.r_[np.geomspace(lo, best, n), np.geomspace(best, hi, n)[1:]]  # lo~best, best~hi를 같은 비율 간격으로
    return sorted({float(f"{v:.3g}") for v in vals})  # 유효숫자 3자리로 반올림, 중복 제거

def best_of(table, grid):  # 표 1등 행의 파라미터 dict (NaN -> None, 숫자는 float)
    r = table.iloc[0]
    return {k: (None if pd.isna(r[k]) else float(r[k])) for k in grid}

def marginal(table, grid, by="logloss"):  # 파라미터마다 값별 가장 좋은 by 지표 (나머지는 그 값에서 가장 좋은 조합) -> 어느 파라미터가 민감한지
    out = {}
    for k in grid:
        s = table.assign(**{k: table[k].astype(object).where(table[k].notna(), "None")}).groupby(k, sort=False)[by].min()
        out[k] = s.reindex([("None" if v is None else v) for v in grid[k] if ("None" if v is None else v) in s.index])
    return out

def game_ci(d, gid, B=1000):  # 경기 단위 부트스트랩 95% 구간 (같은 경기 공은 독립이 아님)
    s = pd.DataFrame({"g": gid, "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])




from scipy.special import log_softmax
for fn in (fc.situation_features, fc.opponent_features, fc.batter_location_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(LOC_TRAIN_FIRST)], n=150)
    assert bad.empty, bad
SIT_L = fc.situation_features(m, fp)
XLOC = pd.concat([fp[CLASSES + ["same_side"]], SIT_L, fc.opponent_features(m, fp)[["home"]], fc.batter_location_features(m, fp)], axis=1)
XLOC["first_open"] = (((SIT_L["base2"] + SIT_L["base3"]) > 0) & SIT_L["base1"].eq(0)).astype(int)  # 득점권인데 1루 비었음
XLOC = XLOC[LOC_FEAT]
NZ = len(ZONES)
INIT_L = np.log(np.clip(PZ_TRUE, 1e-6, None))
INIT_L -= INIT_L.mean(1, keepdims=True)  # 출발점
yr_l = fp["GYEAR"].to_numpy()
BASE_L = dict(objective="multiclass", num_class=NZ, learning_rate=0.05, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
              verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)

def fit_loc(years, lp, rounds=None, val_year=None):  # years 해 초구로 위치 보정 학습 (rounds 없으면 val_year로 조기 종료)
    tr = np.flatnonzero(np.isin(yr_l, years))
    dtr = lgb.Dataset(XLOC.iloc[tr], y_zone[tr], init_score=INIT_L[tr], free_raw_data=False)
    if rounds is None:
        va = np.flatnonzero(yr_l == val_year)
        dva = lgb.Dataset(XLOC.iloc[va], y_zone[va], init_score=INIT_L[va], reference=dtr)
        return lgb.train({**BASE_L, **lp}, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    return lgb.train({**BASE_L, **lp}, dtr, rounds)

iv_l = np.flatnonzero(val)
def score_lgbz(num_leaves, min_data_in_leaf, lambda_l2):  # 한 조합의 2019 24칸 log loss (실제 구종), 트리 수
    lp = {"num_leaves": int(num_leaves), "min_data_in_leaf": int(min_data_in_leaf), "lambda_l2": float(lambda_l2)}
    bst = fit_loc([2018], lp, val_year=fc.VAL_YEAR)
    raw = bst.predict(XLOC.iloc[iv_l], raw_score=True, num_iteration=bst.best_iteration) + INIT_L[iv_l]
    return {"트리 수": bst.best_iteration, "logloss": -log_softmax(raw, axis=1)[np.arange(len(iv_l)), y_zone[iv_l]].mean()}

GRID_LZ = {"num_leaves": [7, 15], "min_data_in_leaf": [20, 50, 200], "lambda_l2": [300.0, 1000.0, 3000.0]}
res_lz = grid_search(score_lgbz, GRID_LZ)
tried_lz = {k: list(v) for k, v in GRID_LZ.items()}
for rnd in range(3):  # 1등이 범위 끝이면 넓힘
    b = res_lz.iloc[0]
    add = {}
    if b.lambda_l2 == max(tried_lz["lambda_l2"]):
        add["lambda_l2"] = [b.lambda_l2 * 3]
    elif b.lambda_l2 == min(tried_lz["lambda_l2"]):
        add["lambda_l2"] = [b.lambda_l2 / 3]
    if b.num_leaves == max(tried_lz["num_leaves"]):
        add["num_leaves"] = [int(b.num_leaves * 2)]
    elif b.num_leaves == min(tried_lz["num_leaves"]):
        add["num_leaves"] = [3]
    if b.min_data_in_leaf == max(tried_lz["min_data_in_leaf"]):
        add["min_data_in_leaf"] = [int(b.min_data_in_leaf * 4)]
    add = {k: v for k, v in add.items() if v[0] not in tried_lz[k]}
    if not add:
        break
    print(f"{rnd + 1}번째 넓히기:", add)
    for k, v in add.items():
        tried_lz[k] += v
        res_lz = pd.concat([res_lz, grid_search(score_lgbz, {**{kk: [b[kk]] for kk in GRID_LZ}, k: v})])
    res_lz = res_lz.drop_duplicates(list(GRID_LZ)).sort_values("logloss").reset_index(drop=True)
TOL = 2e-4
ok_ = res_lz[res_lz["logloss"] <= res_lz["logloss"].iloc[0] + TOL]
pick = ok_.sort_values(["num_leaves", "트리 수"]).iloc[0]  # 평평하면 덜 극단적인 값: 잎 적게 -> 트리 적게
LOC_LGB = {"num_leaves": int(pick.num_leaves), "min_data_in_leaf": int(pick.min_data_in_leaf), "lambda_l2": float(pick.lambda_l2)}
LOC_ROUNDS = int(pick["트리 수"])
ll17_19 = -np.log(np.clip(PZ_TRUE[iv_l][np.arange(len(iv_l)), y_zone[iv_l]], 1e-12, None)).mean()
print(f"2019 24칸 log loss (실제 구종): model_14 {fc.align(m14, fp, fc.VAL_YEAR)['ll_zone_true'].mean():.5f} / model_13 {ll17_19:.5f} / 1등 {res_lz.logloss.iloc[0]:.5f} / 선택 {pick.logloss:.5f} {LOC_LGB}, 트리 {LOC_ROUNDS}개")
display(res_lz.head(10).round(5))
for k, s in marginal(res_lz, {k: sorted(v) for k, v in tried_lz.items()}).items():
    print(k, s.round(5).to_dict())

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개
1번째 넓히기: {'lambda_l2': [np.float64(9000.0)], 'num_leaves': [3]}
2019 24칸 log loss (실제 구종): model_14 2.88354 / model_13 2.88724 / 1등 2.88236 / 선택 2.88236 {'num_leaves': 3, 'min_data_in_leaf': 20, 'lambda_l2': 3000.0}, 트리 565개


,num_leaves,min_data_in_leaf,lambda_l2,트리 수,logloss
0,3.0,20.0,3000.0,565,2.88236
1,7.0,20.0,3000.0,409,2.88241
2,7.0,50.0,3000.0,408,2.88241
3,7.0,20.0,9000.0,1203,2.88243
4,7.0,20.0,1000.0,152,2.88246
5,7.0,200.0,3000.0,409,2.88249
6,7.0,50.0,1000.0,168,2.88250
7,15.0,200.0,3000.0,409,2.88256
8,7.0,200.0,1000.0,168,2.88256
9,15.0,50.0,3000.0,408,2.88257


num_leaves {3: 2.88236, 7: 2.88241, 15: 2.88256}
min_data_in_leaf {20: 2.88236, 50: 2.88241, 200: 2.88249}
lambda_l2 {300.0: 2.88277, 1000.0: 2.88246, 3000.0: 2.88236, 9000.0: 2.88243}


#### 선택: 2019 1등 (잎 3 · 잎당 최소 20 · lambda_l2 3000 · 트리 565개)
- 2019 24칸 log loss (실제 구종): model_14 2.88354 -> 2.88236 (−0.0012)
  - 파라미터도 바뀜 (model_14: 잎 7 · λ 1000 · 트리 168) -> model_14 격자 1등(잎 15 · λ 3000, 2.88334)과 비교해도 −0.0010 -> 이득 대부분은 장타 비율 몫
- 잎 3은 넓혀서 들어간 값이고 다시 범위 끝 (잎 7과 차이 0.00005, 평평) -> 잎 2(가지 하나)는 확인 안 함
- λ 3000 (9000은 나쁨), 잎당 최소 공은 평평

### 3-3. 최종 학습과 진단

- 2019 = 2018 학습, 2020 = 2018–19 학습 (파라미터·트리 수는 3-2 값)
- 진단 (실제 구종 기준): 주자 상황·아웃별 존 밖(C) 실제 − 예측 (model_13 = 보정 없음 vs model_15), 존 밖 AUC, 변수 중요도

In [11]:
BST_L = {fc.VAL_YEAR: fit_loc([2018], LOC_LGB, LOC_ROUNDS), fc.TEST_YEAR: fit_loc([2018, 2019], LOC_LGB, LOC_ROUNDS)}

def loc_correct(P, q, year, t=None):  # model_13 24칸 확률 P (n, 24) -> 상황 보정 확률, t = 가정한 구종 (None이면 q의 실제 구종)
    X = XLOC.loc[q.index].copy()
    if t is not None:
        X.loc[:, CLASSES] = np.tile((np.array(CLASSES) == t).astype(float), (len(X), 1))  # 구종 원-핫을 t로
    init = np.log(np.clip(P, 1e-6, None))
    init -= init.mean(1, keepdims=True)
    return np.exp(log_softmax(BST_L[year].predict(X, raw_score=True) + init, axis=1))

def runner3(q):
    return np.select([(q["base2"] + q["base3"]) > 0, q["base1"] > 0], ["득점권", "1루만"], "없음")

rows, imp = [], None
for yr, q, msk in [(fc.VAL_YEAR, q_val, val), (fc.TEST_YEAR, q_test, test)]:
    i, yz = np.arange(len(q)), y_zone[msk]
    P17 = PZ_TRUE[msk]
    P18 = loc_correct(P17, q, yr)
    oc = (fc.ZONE_GROUP[yz] == "C").astype(float)
    d = pd.DataFrame({"주자": runner3(SIT_L.loc[q.index]), "아웃": SIT_L.loc[q.index, "out_cn"].clip(upper=2).to_numpy(), "C 실제": oc,
                      "C 17": P17[:, fc.ZONE_GROUP == "C"].sum(1), "C 18": P18[:, fc.ZONE_GROUP == "C"].sum(1),
                      "이득": (np.log(np.clip(P18[i, yz], 1e-12, None)) - np.log(np.clip(P17[i, yz], 1e-12, None))) * 1000})
    for var in ["주자", "아웃"]:
        for v, g in d.groupby(var):
            rows.append({"연도": yr, "변수": var, "값": str(v), "공": len(g), "rC model_13": (g["C 실제"] - g["C 17"]).mean(),
                         "rC model_15": (g["C 실제"] - g["C 18"]).mean(), "이득 ×1000": g["이득"].mean()})
    rows.append({"연도": yr, "변수": "전체", "값": "", "공": len(d), "rC model_13": (d["C 실제"] - d["C 17"]).mean(),
                 "rC model_15": (d["C 실제"] - d["C 18"]).mean(), "이득 ×1000": d["이득"].mean(),
                 "존 밖 AUC 17": auc(d["C 17"].to_numpy(), oc), "존 밖 AUC 19": auc(d["C 18"].to_numpy(), oc)})
display(pd.DataFrame(rows).set_index(["변수", "값", "연도"]).unstack("연도").round(4))
gain = pd.Series(BST_L[fc.VAL_YEAR].feature_importance("gain"), index=LOC_FEAT)
print("변수 중요도 (gain 비율, 2019 모델):", (gain / gain.sum()).sort_values(ascending=False).round(3).to_dict())

공        rC model_13         rC model_15         이득 ×1000         존 밖 AUC 17         존 밖 AUC 19        
연도       2019   2020        2019    2020        2019    2020     2019    2020       2019    2020       2019    2020
변수 값                                                                                                               
아웃 0     4645   4375     -0.0324 -0.0260     -0.0200 -0.0060   9.8103  3.9095        NaN     NaN        NaN     NaN
   1     4500   4222     -0.0079 -0.0148     -0.0022 -0.0056   3.2499  6.0300        NaN     NaN        NaN     NaN
   2     4313   4018     -0.0106  0.0168     -0.0131  0.0193   1.2857  0.4805        NaN     NaN        NaN     NaN
전체      13458  12615     -0.0173 -0.0086     -0.0118  0.0022   4.8847  3.5270     0.5483  0.5428     0.5543  0.5505
주자 1루만   2886   2723     -0.0228  0.0026     -0.0311  0.0040   2.6521  0.8838        NaN     NaN        NaN     NaN
   득점권   3346   3136      0.0367  0.0266      0.0325  0.0183   4.2709  4.8350        NaN     NaN        NaN     NaN
   없음    7226   6756     -0.0400 -0.0295     -0.0247 -0.0060   6.0606  3.9852        NaN     NaN        NaN     NaN

변수 중요도 (gain 비율, 2019 모델): {'bat_xbh': 0.315, 'batorder': 0.125, 'out_cn': 0.124, 'sdiff': 0.095, 'inning': 0.088, 'base1': 0.078, 'FAST': 0.034, 'home': 0.032, 'base2': 0.027, 'same_side': 0.024, 'CHUP': 0.013, 'first_open': 0.012, 'base3': 0.008, 'SLID': 0.008, 'CURV': 0.007, 'FORK': 0.004, 'TWOS': 0.004, 'CUTT': 0.0, 'KNUC': 0.0, 'SINK': 0.0}


#### 🔍 3-3 결과: 2019는 더 좋아지지만 존 밖 치우침은 model_14과 거의 같음
- 전체 이득 ×1000 (실제 구종, 보정 없음 대비): 2019 +4.88 (model_14 +3.70), 2020 +3.53 (model_14 +3.25)
- 존 밖 AUC: 2019 0.554 (model_14 0.555), 2020 0.551 (0.552) -> 존 안/밖 구분은 늘지 않음
- 주자·아웃별 존 밖 치우침은 model_14과 거의 같음
- 변수 중요도: 장타 비율 0.32로 가장 큼 (그다음 타순 0.13, 아웃 0.12) -> 트리가 많이 쓰지만 이득은 작음

## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_12과 같음, 위치만 3-1에서 고른 값 (분산 따로 포함)

In [12]:
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, LOC_SEASON_F, LOC_AC, LOC_VAR)
        P = loc_correct(P, q, year, t)  # 상황 보정 (model_14), 정규분포 g는 model_13 그대로
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [13]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_TYPE_ALL[val]), predict(q_test, fc.TEST_YEAR, P_TYPE_ALL[test])], ignore_index=True)
print(pp.shape)
# 확인: 구종·좌표(정규분포)는 model_14과 같아야 함, 24칸만 바뀜
chk = pp.merge(m14, on=fc.KEYS, suffixes=("", "_14"))
same = ["ll_type", "nll_coord_true", "dist_true"] + fc.P_TYPE
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_14"], equal_nan=True, atol=1e-6)]
assert len(chk) == len(pp) and not bad, f"model_14과 다른 열: {bad}"
print("구종·좌표 = model_14 (재현 확인), 24칸 확률만 바뀜")

(26073, 48)
구종·좌표 = model_14 (재현 확인), 24칸 확률만 바뀜


## 5. 평가

- 채점은 `fc.evaluate`, model_14과 나란히 / 경기 단위 부트스트랩 (model_15 − model_14, 2020)
- 2020은 확인용, 너클 뺀 값도 봄 (4-27)
- 장타 비율 구간·타순별 이득 (model_14 대비, 실제 구종 기준): 강타자에게 존 밖으로 피하는 게 맞는지

In [14]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m14_sum = eval_years(m14)
display(pd.concat({MODEL: summary, "model_14": m14_sum, "차이 (model_14 대비)": summary - m14_sum}, axis=1).round(4))

model_15          model_14          차이 (model_14 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0613   1.0561   1.0613   1.0561           0.0000  0.0000
구종 top1               0.5385   0.5332   0.5385   0.5332           0.0000  0.0000
구종 top2               0.8177   0.8218   0.8177   0.8218           0.0000  0.0000
구종 top3               0.9374   0.9412   0.9374   0.9412           0.0000  0.0000
구종 macro F1           0.3226   0.3545   0.3226   0.3545           0.0000  0.0000
24칸 logloss (실전)      2.9418   2.9398   2.9429   2.9403          -0.0011 -0.0005
24칸 top1              0.1335   0.1328   0.1329   0.1328           0.0006  0.0000
24칸 top3              0.2980   0.2993   0.2951   0.2978           0.0030  0.0015
H/S/C logloss         1.0738   1.0745   1.0736   1.0744           0.0002  0.0001
H/S/C 정확도             0.4005   0.3992   0.4014   0.3983          -0.0009  0.0010
24칸 logloss (실제 구종)   2.8824   2.8836   2.8835   2.8839          -0.0012 -0.0003
구종×칸 logloss          3.9437   3.9397   3.9448   3.9399          -0.0012 -0.0003
좌표 평균거리 cm           32.6377  31.8198  32.6377  31.8198           0.0000  0.0000
좌표 NLL                9.3124   9.2662   9.3124   9.2662           0.0000  0.0000

In [15]:
display(fc.bootstrap_diff(fc.pitch_scores(m14, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = 이 모델이 좋음 (b − a)
rows = []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    a18, a19 = fc.align(m14, fp, yr), fc.align(pp, fp, yr)
    nk = a18["pitch_type"].ne("KNUC").to_numpy()
    d_true = a19["ll_zone_true"].to_numpy() - a18["ll_zone_true"].to_numpy()
    print(f"{yr} 24칸 실제 구종 차이 (model_15 − model_14): 전체 {d_true.mean():+.5f}, 너클 뺌 {d_true[nk].mean():+.5f}")
    k = fp["GYEAR"].eq(yr).to_numpy()
    xbh = XLOC.loc[k, "bat_xbh"].to_numpy()
    oc = a18["zone_group"].eq("C").to_numpy().astype(float)
    c18 = a18[[f"p_zone_{z}" for z in ZONES]].to_numpy()[:, fc.ZONE_GROUP == "C"].sum(1)  # 실전 C 확률
    c19 = a19[[f"p_zone_{z}" for z in ZONES]].to_numpy()[:, fc.ZONE_GROUP == "C"].sum(1)
    d = pd.DataFrame({"장타 비율": pd.qcut(xbh, 5, labels=["하위 20%", "20–40%", "40–60%", "60–80%", "상위 20%"]),
                      "타순": SIT_L.loc[k, "batorder"].map(lambda b: "1–2번" if b <= 2 else "3–5번" if b <= 5 else "6–9번").to_numpy(),
                      "이득": -d_true * 1000, "C 실제": oc, "C 18": c18, "C 19": c19})
    for var in ["장타 비율", "타순"]:
        for v, g in d.groupby(var, observed=True):
            rows.append({"연도": yr, "변수": var, "값": str(v), "공": len(g), "이득 ×1000 (실제 구종)": g["이득"].mean(),
                         "C 실제": g["C 실제"].mean(), "C 예측 18 (실전)": g["C 18"].mean(), "C 예측 19 (실전)": g["C 19"].mean()})
display(pd.DataFrame(rows).set_index(["변수", "값", "연도"]).unstack("연도").round(4))

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,0.0000,0.0000,0.0000,차이 불확실
ll_zone,-0.0005,-0.0018,0.0007,차이 불확실
ll_joint,-0.0003,-0.0016,0.0009,차이 불확실
nll_coord,0.0000,0.0000,0.0000,차이 불확실
dist,0.0000,0.0000,0.0000,차이 불확실


2019 24칸 실제 구종 차이 (model_15 − model_14): 전체 -0.00118, 너클 뺌 -0.00118
2020 24칸 실제 구종 차이 (model_15 − model_14): 전체 -0.00028, 너클 뺌 -0.00024


공       이득 ×1000 (실제 구종)            C 실제         C 예측 18 (실전)         C 예측 19 (실전)        
연도            2019  2020             2019    2020    2019    2020         2019    2020         2019    2020
변수    값                                                                                                    
장타 비율 20–40%  2691  2524           1.7442 -2.5087  0.3935  0.3756       0.4009  0.3820       0.4006  0.3844
      40–60%  2693  2522          -0.1414 -1.1597  0.3932  0.3779       0.4011  0.3828       0.4037  0.3841
      60–80%  2691  2523          -1.9219 -1.5906  0.4039  0.3908       0.4057  0.3854       0.4054  0.3881
      상위 20%  2691  2523           2.9163  7.3214  0.3980  0.3960       0.4117  0.3878       0.4163  0.3878
      하위 20%  2692  2523           3.3119 -0.6672  0.3737  0.3864       0.3996  0.3789       0.3941  0.3724
타순    1–2번    3240  3055           2.1663  0.5254  0.3735  0.3663       0.3960  0.3726       0.3959  0.3725
      3–5번    4589  4371           1.0199  1.6565  0.4081  0.4075       0.4119  0.3915       0.4120  0.3927
      6–9번    5629  5189           0.7471 -1.0266  0.3907  0.3779       0.4016  0.3829       0.4022  0.3819

### 🥇1등🥇
#### 2019 −0.0012였지만 2020 실제 구종 −0.0003 (실전 −0.0005, 모두 차이 불확실) -> 채택 안 함
- 2019 24칸 실제 구종 −0.0012 / 실전 −0.0011, 2020 실제 구종 −0.0003 (너클 빼면 −0.0002) / 실전 −0.0005
- 부트스트랩 (2020, model_15 − model_14): 6개 점수 모두 차이 불확실, 나빠짐 없음 -> 형식상 채택 조건은 만족
- 채택 안 한 이유: 2020 크기가 2019의 1/4, 0장에서 타자 변수 4개 중 하나만 겨우 + (우연 가능성), 장타 비율 구간별 이득이 두 해에 일관되지 않음 (중간 구간 2020 −1.2 ~ −2.5), 존 밖 AUC 늘지 않음
- 잎 3이 범위 끝 (잎 2 확인 안 함), 확인해도 판단은 같을 것으로 봄
- **채택 안 함 -> model_14이 계속 비교 출발점** (사용자 결정 2026-11-11)

## 6. 저장

- `results/model_15/` (채택하면 `compare.ipynb` `ADOPTED`에 추가)

In [16]:
fc.save_results(MODEL, pp, {"summary": summary})

저장: results\model_15 ['per_pitch.parquet', 'summary.csv']
